# Semester Project — Estimation of Obesity Levels
## Part 2 — Supervised Learning Baseline
### File 1: Data Loading + Naive Bayes (Naeem Akmal)

**Group:** Muhaimin Azeem | Naeem Akmal | Rana Fahad Ahmad  
**Course:** Introduction to Machine Learning  
**Dataset:** UCI Estimation of Obesity Levels (2,111 rows, 16 features)  
**Target:** NObeyesdad (7-class classification)

---
**This file covers:**
1. Load processed data from Drive (Part 1 checkpoint)
2. Train Gaussian Naive Bayes
3. Manual Information Gain calculation (hand-worked example)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive

from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import label_binarize

drive.mount('/content/drive')
sns.set_style('whitegrid')
%matplotlib inline
print('Imports done ✓')

In [ ]:
# ── Load processed arrays saved in Part 1 ──────────────────────────────────
X_train = np.load('/content/drive/MyDrive/X_train_processed.npy')
X_test  = np.load('/content/drive/MyDrive/X_test_processed.npy')
y_train = np.load('/content/drive/MyDrive/y_train.npy', allow_pickle=True)
y_test  = np.load('/content/drive/MyDrive/y_test.npy',  allow_pickle=True)

print('X_train:', X_train.shape)
print('X_test :', X_test.shape)
print('y_train:', y_train.shape, '| unique classes:', len(np.unique(y_train)))
print('Classes:', np.unique(y_train))

## 1. Naive Bayes — Theory

**Naive Bayes** is a probabilistic classifier based on Bayes' Theorem:

$$P(y \mid x_1, \ldots, x_n) \propto P(y) \prod_{i=1}^{n} P(x_i \mid y)$$

The *naive* assumption: all features are **conditionally independent** given the class.

**Gaussian NB** models each feature as a Gaussian distribution per class:

$$P(x_i \mid y) = \frac{1}{\sqrt{2\pi\sigma_y^2}} \exp\!\left(-\frac{(x_i - \mu_y)^2}{2\sigma_y^2}\right)$$

**Why useful here?**
- Very fast to train (O(n × d))
- Works well as a quick baseline
- Handles multi-class natively

**Limitation:** The independence assumption is often violated in real data (e.g. Weight and BMI are correlated).

In [ ]:
# ── Train Gaussian Naive Bayes ──────────────────────────────────────────────
gnb = GaussianNB()
gnb.fit(X_train, y_train)

y_pred_nb = gnb.predict(X_test)

nb_acc = accuracy_score(y_test, y_pred_nb)
nb_f1  = f1_score(y_test, y_pred_nb, average='macro')

print(f'Naive Bayes  |  Accuracy: {nb_acc:.4f}  |  Macro F1: {nb_f1:.4f}')
print()
print(classification_report(y_test, y_pred_nb))

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(9, 7))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_nb,
    ax=ax,
    xticks_rotation=45,
    colorbar=False
)
ax.set_title('Naive Bayes — Confusion Matrix (Test Set)', fontsize=13)
plt.tight_layout()
plt.show()

## 2. Manual Information Gain Calculation

Decision Trees split on features that give the highest **Information Gain (IG)**:

$$IG(S, A) = H(S) - \sum_{v \in \text{values}(A)} \frac{|S_v|}{|S|} H(S_v)$$

Where entropy $H$ is:

$$H(S) = -\sum_{c} p_c \log_2 p_c$$

We demonstrate this on a **small 10-row sample** drawn from the dataset so the maths is tractable by hand.  
Feature chosen: **FAVC** (Frequent consumption of high caloric food) — binary: yes / no.

In [ ]:
# ── Manual Information Gain: FAVC vs NObeyesdad ─────────────────────────────
# We reload the raw CSV just for this demonstration

df_raw = pd.read_csv('/content/drive/MyDrive/ObesityDataSet_raw_and_data_sinthetic.csv')

# Take a fixed 10-row sample (same every run)
sample = df_raw[['FAVC', 'NObeyesdad']].sample(n=10, random_state=7).reset_index(drop=True)
print('Sample used for manual IG calculation:')
print(sample.to_string())

# ── Step 1: Entropy of the whole sample (S) ────────────────────────────────
def entropy(labels):
    """Shannon entropy in bits."""
    n = len(labels)
    if n == 0:
        return 0.0
    counts = pd.Series(labels).value_counts()
    probs  = counts / n
    return float(-np.sum(probs * np.log2(probs + 1e-12)))

H_S = entropy(sample['NObeyesdad'])
print(f'\nStep 1 — Entropy of full sample  H(S) = {H_S:.4f} bits')

# ── Step 2: Weighted entropy after splitting on FAVC ──────────────────────
print('\nStep 2 — Split on FAVC (yes / no):')
weighted_H = 0.0
for val in sample['FAVC'].unique():
    subset = sample[sample['FAVC'] == val]['NObeyesdad']
    h_sub  = entropy(subset)
    weight = len(subset) / len(sample)
    weighted_H += weight * h_sub
    print(f'  FAVC={val}: n={len(subset)}, H={h_sub:.4f}, weight={weight:.2f}')

print(f'\n  Weighted entropy H(S|FAVC) = {weighted_H:.4f} bits')

# ── Step 3: Information Gain ───────────────────────────────────────────────
IG = H_S - weighted_H
print(f'\nStep 3 — IG(S, FAVC) = {H_S:.4f} - {weighted_H:.4f} = {IG:.4f} bits')
print('\nConclusion: A decision tree would consider splitting on FAVC to be',
      f'{"useful" if IG > 0.1 else "relatively uninformative"} at this node.')

In [ ]:
# ── Information Gain for all categorical features (full dataset) ────────────
cat_features = ['Gender', 'family_history_with_overweight', 'FAVC',
                'CAEC', 'SMOKE', 'SCC', 'CALC', 'MTRANS']

H_full = entropy(df_raw['NObeyesdad'])
ig_scores = {}
for feat in cat_features:
    wh = 0.0
    for val in df_raw[feat].unique():
        sub = df_raw[df_raw[feat] == val]['NObeyesdad']
        wh += (len(sub) / len(df_raw)) * entropy(sub)
    ig_scores[feat] = H_full - wh

ig_df = pd.DataFrame.from_dict(ig_scores, orient='index', columns=['IG']).sort_values('IG', ascending=True)

ig_df.plot(kind='barh', legend=False, color='steelblue', figsize=(8, 4))
plt.xlabel('Information Gain (bits)')
plt.title('Information Gain per Categorical Feature vs NObeyesdad')
plt.tight_layout()
plt.show()

print('\nIG scores:')
print(ig_df.sort_values('IG', ascending=False).round(4))

In [ ]:
import pickle

with open('/content/drive/MyDrive/gnb_model.pkl', 'wb') as f:
    pickle.dump(gnb, f)

# Also save predictions for cross-notebook evaluation
np.save('/content/drive/MyDrive/y_pred_nb.npy', y_pred_nb)

print('Naive Bayes model saved to Drive ✓')
print(f'Final  →  Accuracy: {nb_acc:.4f}  |  Macro F1: {nb_f1:.4f}')